# U.S. reserved evaluation: approval required

Do not execute until Anna and Zasha explicitly approve `docs/us_reserved_test_protocol.md` and the frozen model hash. This notebook has not been run. The 11,475 reserved campaigns include 1,486 IDs exposed in historical development. Report retrospective evidence, not an independent holdout. Do not retrain, tune or change the threshold after scoring.

In [ ]:
from pathlib import Path
import sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils/preprocessing.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Fill these only after explicit team approval of the protocol and artifact.
APPROVAL = {'anna': '', 'zasha': '', 'date': '', 'artifact_sha256': ''}
if not all(APPROVAL.values()):
    raise RuntimeError('Stop: explicit team approval and frozen model hash are required')
RESULT = ROOT / 'models/us_reserved_evaluation_v1.json'
if RESULT.exists():
    raise RuntimeError('A reserved evaluation already exists. Do not rerun or tune from it.')
MODEL = ROOT / 'models/us_primary_model_v1.joblib'
actual_hash = hashlib.sha256(MODEL.read_bytes()).hexdigest()
if actual_hash != APPROVAL['artifact_sha256']:
    raise RuntimeError('Approved artifact hash does not match')
from utils.us_prediction import USPredictor
predictor = USPredictor(MODEL)
if predictor.metadata['artifact_sha256'] != actual_hash or predictor.threshold != 0.5:
    raise RuntimeError('Frozen metadata/hash/threshold mismatch')

The following cell opens reserved labels. The approval gate above must succeed first. Every assertion is checked before computing metrics. The raw snapshot is used only to replay historical exposure membership.

In [ ]:
if not all(APPROVAL.values()):
    raise RuntimeError('Stop: explicit team approval is required before reserved reads')
if RESULT.exists() or hashlib.sha256(MODEL.read_bytes()).hexdigest() != APPROVAL['artifact_sha256']:
    raise RuntimeError('Stop: evaluation already exists or frozen artifact changed')
import pandas as pd
import numpy as np
from utils.us_validation import INPUTS, metrics
from utils.utils import get_data_split
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
US = ROOT / 'data/Processed-Data/us_only'
train_ids = pd.read_csv(US / 'train_project_ids.csv', usecols=['project id'])
reserved = pd.read_csv(US / 'test_project_ids.csv')
assert len(reserved) == 11475 and reserved['project id'].is_unique
assert set(train_ids['project id']).isdisjoint(reserved['project id'])
labels = pd.read_csv(US / 'y_test.csv')['target']
assert len(labels) == 11475 and labels.isin([0, 1]).all()
assert np.array_equal(labels.to_numpy(), reserved['target'].to_numpy())
clean = pd.read_csv(US / 'DSI_kickstarterscrape_dataset_cleaned.csv', usecols=['project id', *INPUTS])
assert clean['project id'].is_unique
X_reserved = clean.set_index('project id').loc[reserved['project id'], INPUTS].reset_index(drop=True)
raw_path = ROOT / 'data/Raw-data/DSI_kickstarterscrape_dataset.csv'
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == 'b788f3c4527c8203bd237367b2c142046f5dbf9bd48fe5d84ca9478126ab8c52'
raw = pd.read_csv(raw_path, encoding_errors='replace')
historical_train, _, _, _ = get_data_split(raw)
exposed = reserved['project id'].isin(historical_train['project id']).to_numpy()
assert exposed.sum() == 1486 and (~exposed).sum() == 9989
probability = predictor.pipeline.predict_proba(X_reserved)[:, 1]
results = {}
for name, mask in {'all': np.ones(len(labels), dtype=bool),
                   'historically_exposed': exposed, 'not_known_exposed': ~exposed}.items():
    results[name] = dict(n=int(mask.sum()), **metrics(labels[mask], probability[mask], predictor.threshold))
display(pd.DataFrame(results).T.round(4))
ConfusionMatrixDisplay.from_predictions(labels, probability >= predictor.threshold,
    labels=[0, 1], display_labels=['Failed', 'Successful'], cmap='Greens', colorbar=False)
plt.tight_layout(); plt.show()
report = dict(approval=APPROVAL, artifact_sha256=actual_hash, threshold=predictor.threshold,
              metrics=results, limitations=predictor.metadata['limitations'],
              interpretation='Retrospective historically exposed holdout; not-known-exposed group is not established independent')
with RESULT.open('x') as output:
    json.dump(report, output, indent=2, allow_nan=False)
print('Saved one reserved evaluation:', RESULT)